In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install necessary packages (run once at the start of your notebook/script)
!pip install mne plotly colorama --upgrade
!pip install mne
# Import libraries
import os
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from colorama import Fore, Style, init
import plotly.graph_objects as go
import gc
import traceback

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 104.5 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1


In [ ]:
# Root EEG data directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

**Project Title:** Distal Speech Rate ERP Analysis - Summer 2025

**Principal Investigator:**
  - Prof. [Your PI Name], [Your Institution]

**Research Team:**

- [Your Name], [Your Institution]
- [Your Name], [Your Institution]
- Dr. Lisa Sanders, University of Massachusetts Amherst (UMass)
- Yimeng Wang, University of Massachusetts Amherst (UMass)


**Institution(s):**
  Mount Holyoke College, University of Massachusetts Amherst (UMass)

**Date:** August 2025

**Introduction:**

This script processes EEG/ERP data to examine how distal speech rate influences early perceptual processing of function words with weak acoustic onsets. The analysis focuses on extracting N1 (100-150 ms) and N280 (200-300 ms) components to assess differences in neural responses based on speech rate context.

Based on: Sanders et al., "Distal Speech Rate Makes Words Disappear from Early Perceptual Processing"

**PREPROCESSING** - Participnats :
                ['3', '6', '7', '8', '9', '10', '11', '12' '13', '14',
                '15', '17', '18', '19', '20', '21', '22', '23',
                '24', '25', '26', '27']

# **PROCEDURE**

1. Annotate the breaks in one .set file for each subject
2. Filtering
- 60 hz line noise
- 0.1-1 hz high pass
- 40-50 hz low pass
3. Bad channels detection for all the sessions
4. Perform ICA - plot all components
5. Re-reference (LM/RM)
6. Create Epochs
7. Artifacts Rejection - get the clean epochs
8. Average within each subject
 - Subject-level ERP (all E trials)
 - Subject-level ERP (all N trials)
9. Average across subjects  → Grand average ERPs
- Grand average ERP (all E trials)
 - Grand average ERP (all N trials)
10. Create 9 Region plots
11. Statistical Analysis


# Data Preprocessing:



# 1. Annotate the breaks in one .set file for each subject

In [ ]:
# Step 3: Set paths and parameters
import os
import numpy as np
import pandas as pd
import mne

root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
gap_threshold = 100.0  # seconds
summary_records = []

# Step 4: Loop through participant folders
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for pid in participants:
    set_path = os.path.join(root_dir, pid, f"{pid}_001.set")
    fif_path = os.path.join(root_dir, pid, f"{pid}_annotated_raw.fif")

    # ✅ Skip participant if annotated file already exists
    if os.path.exists(fif_path):
        print(f"⏭️  {pid}: annotated file already exists — skipping.")
        continue

    if not os.path.exists(set_path):
        print(f"[!] {pid}: .set file not found.")
        continue

    print(f"\n=== Processing Participant {pid} ===")

    try:
        raw = mne.io.read_raw_eeglab(set_path, preload=True)
    except Exception as e:
        print(f"[ERROR loading {pid}]: {e}")
        continue

    # Step 5: Detect long gaps between annotations
    onsets = raw.annotations.onset
    gaps = np.diff(onsets)
    long_gap_indices = np.where(gaps > gap_threshold)[0]

    print(f"→ Found {len(long_gap_indices)} long gaps > {gap_threshold}s")

    for idx in long_gap_indices:
        onset = onsets[idx]
        duration = gaps[idx]
        raw.annotations.append(onset, duration, 'break')
        summary_records.append({
            'participant': pid,
            'onset_sec': round(onset, 3),
            'duration_sec': round(duration, 3)
        })

    # Step 6: Save annotated raw file
    raw.save(fif_path, overwrite=True)
    print(f"✓ Saved annotated raw to: {fif_path}")

# Step 7: Save CSV summary of breaks
summary_df = pd.DataFrame(summary_records)
csv_output_path = os.path.join(root_dir, 'break_summary.csv')
summary_df.to_csv(csv_output_path, index=False)
print(f"\n📁 Summary CSV saved to: {csv_output_path}")

# Optional: Display first few rows
summary_df.head()


⏭️  10: annotated file already exists — skipping.
⏭️  11: annotated file already exists — skipping.
⏭️  12: annotated file already exists — skipping.
⏭️  13: annotated file already exists — skipping.
⏭️  14: annotated file already exists — skipping.
⏭️  15: annotated file already exists — skipping.
⏭️  17: annotated file already exists — skipping.
⏭️  18: annotated file already exists — skipping.
⏭️  19: annotated file already exists — skipping.
⏭️  20: annotated file already exists — skipping.
⏭️  21: annotated file already exists — skipping.
⏭️  22: annotated file already exists — skipping.
⏭️  24: annotated file already exists — skipping.
⏭️  25: annotated file already exists — skipping.
⏭️  26: annotated file already exists — skipping.
⏭️  27: annotated file already exists — skipping.
⏭️  3: annotated file already exists — skipping.
⏭️  6: annotated file already exists — skipping.
⏭️  7: annotated file already exists — skipping.
⏭️  8: annotated file already exists — skipping.
[!] 

""


# 2. Filtering
### - 60 hz line noise
###- 0.1-1 hz high pass
###- 40-50 hz low pass

In [ ]:
# Step 3: Set paths and parameters
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
gap_threshold = 100.0  # seconds
summary_records = []

# Step 4: Loop through participant folders
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for pid in participants:
    set_path = os.path.join(root_dir, pid, f"{pid}_001.set")
    fif_path = os.path.join(root_dir, pid, f"{pid}_annotated_raw.fif")

    if not os.path.exists(set_path):
        print(f"[!] {pid}: .set file not found.")
        continue

    print(f"\n=== Processing Participant {pid} ===")

    try:
        raw = mne.io.read_raw_eeglab(set_path, preload=True)
    except Exception as e:
        print(f"[ERROR loading {pid}]: {e}")
        continue

    # Step 5: Detect long gaps between annotations
    onsets = raw.annotations.onset
    gaps = np.diff(onsets)
    long_gap_indices = np.where(gaps > gap_threshold)[0]

    print(f"→ Found {len(long_gap_indices)} long gaps > {gap_threshold}s")

    for idx in long_gap_indices:
        onset = onsets[idx]
        duration = gaps[idx]
        raw.annotations.append(onset, duration, 'break')
        summary_records.append({
            'participant': pid,
            'onset_sec': round(onset, 3),
            'duration_sec': round(duration, 3)
        })

    # Step 6: Save annotated raw file
    raw.save(fif_path, overwrite=True)
    print(f"✓ Saved annotated raw to: {fif_path}")

# Step 7: Save CSV summary of breaks
summary_df = pd.DataFrame(summary_records)
csv_output_path = os.path.join(root_dir, 'break_summary.csv')
summary_df.to_csv(csv_output_path, index=False)
print(f"\n📁 Summary CSV saved to: {csv_output_path}")

# Show first few rows
summary_df.head()

from google.colab import sheets
sheet = sheets.InteractiveSheet(df=summary_df)


=== Processing Participant 10 ===
Reading /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/10/10_001.fdt
Reading 0 ... 929921  =      0.000 ...  3719.684 secs...
→ Found 5 long gaps > 100.0s
Overwriting existing file.
Writing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/10/10_annotated_raw.fif
Closing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/10/10_annotated_raw.fif
[done]
✓ Saved annotated raw to: /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/10/10_annotated_raw.fif

=== Processing Participant 11 ===
Reading /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/11/11_001.fdt
Reading 0 ... 636442  =      0.000 ...  2545.768 secs...
→ Found 3 long gaps > 100.0s
Overwriting existing file.
Writing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/11/11_annotated_raw.fif
Closing /content/drive/MyDrive/2025_Summer (D

/tmp/ipython-input-12-658239939.py:20: RuntimeWarning: Data file name in EEG.data (3-second.fdt) is incorrect, the file name must have changed on disk, using the correct file name (3_001.fdt).
  raw = mne.io.read_raw_eeglab(set_path, preload=True)


→ Found 3 long gaps > 100.0s
Overwriting existing file.
Writing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/3/3_annotated_raw.fif
Closing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/3/3_annotated_raw.fif
[done]
✓ Saved annotated raw to: /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/3/3_annotated_raw.fif

=== Processing Participant 6 ===
Reading /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/6/6_001.fdt
Reading 0 ... 806596  =      0.000 ...  3226.384 secs...
→ Found 3 long gaps > 100.0s
Overwriting existing file.
Writing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/6/6_annotated_raw.fif
Closing /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/6/6_annotated_raw.fif
[done]
✓ Saved annotated raw to: /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/6/6_annotated_raw.fif

=== Pro

In [ ]:
import mne

fif_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/10/10_annotated_raw.fif'
raw = mne.io.read_raw_fif(fif_path, preload=True)
print(raw.annotations)  # Should show a list of annotations


Opening raw data file /content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/10/10_annotated_raw.fif...
    Range : 0 ... 929921 =      0.000 ...  3719.684 secs
Ready.
Reading 0 ... 929921  =      0.000 ...  3719.684 secs...
<Annotations | 508 segments: CELL (6), CRTE (40), CRTN (39), SESS (2), ...>


# 3. Bad channels detection

In [ ]:
import os
import numpy as np
import pandas as pd
import mne

# Load break summary
break_summary_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/break_summary.csv'
break_df = pd.read_csv(break_summary_path)

# Define regions (same as your list)
regions = { ... }  # <-- keep your region dictionary here

root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Starting Participant {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_O1_filtered_raw.fif')])
    subj_breaks = break_df[break_df['participant'] == int(subj)]

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)

        try:
            raw = mne.io.read_raw_fif(session_path, preload=True)
            raw.set_montage('GSN-HydroCel-128')
            raw.info["bads"] = []
        except Exception as e:
            print(f"[Error reading {session_file}]: {e}")
            continue

        break_list = [(row.onset_sec, row.onset_sec + row.duration_sec) for _, row in subj_breaks.iterrows()]
        file_end = raw.times[-1]

        # === Find 5-minute clean window
        t0 = 0
        found_window = False
        while t0 + 300 <= file_end:
            window_ok = True
            for onset, offset in break_list:
                if onset < t0 + 300 and offset > t0:
                    window_ok = False
                    break
            if window_ok:
                found_window = True
                break
            t0 += 10

        if not found_window:
            print(f"⏩ No 5-min clean window found for {subj} in {session_file}")
            continue

        print(f"✅ Plotting clean window: {t0/60:.2f}–{(t0+300)/60:.2f} min")

        for region_name, region_channels in regions.items():
            # === Define output path to check if plot exists
            plot_path = os.path.join(subj_path, f"{session_file}_{region_name}_preview.png")

            if os.path.exists(plot_path):
                print(f"⏭️  {region_name} already processed for {session_file} — skipping.")
                continue

            picks = [ch for ch in region_channels if ch in raw.ch_names]
            if not picks:
                print(f"No valid channels for {region_name}. Skipping.")
                continue

            # Add dummy channels if < 20
            dummy_index = 1
            while len(picks) < 20:
                dummy_name = f'dummy{dummy_index}'
                if dummy_name not in raw.ch_names:
                    dummy_data = np.zeros((1, raw.n_times))
                    dummy_info = mne.create_info([dummy_name], raw.info['sfreq'], ch_types='eeg')
                    dummy_raw = mne.io.RawArray(dummy_data, dummy_info)
                    raw.add_channels([dummy_raw], force_update_info=True)
                picks.append(dummy_name)
                dummy_index += 1

            # === Plot and save to file
            fig = raw.plot(
                picks=picks,
                n_channels=20,
                scalings=3e-4,
                start=t0,
                duration=30,
                title=f"{session_file} - {region_name} (clean {t0/60:.2f}–{(t0+300)/60:.2f} min)",
                group_by='original',
                show_scrollbars=False,
                show_scalebars=False,
                show=False  # Don't display, just save
            )

            fig.savefig(plot_path)
            print(f"🖼️  Saved: {plot_path}")
            plt.close(fig)


EmptyDataError: No columns to parse from file

# CORRELATION MATRIX

In [ ]:

root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
correlation_threshold = 0.2

subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Processing Subject {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_O1_filtered_raw.fif')])
    print(f"🔍 Found files: {session_files}")

    for session_file in session_files:
        session_path = os.path.join(subj_path, session_file)
        print(f"📂 Loading: {session_path}")
        raw = mne.io.read_raw_fif(session_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')
        raw.info["bads"] = []

        picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
        if not len(picks):
            print("⚠ No EEG channels found. Skipping.")
            continue

        data, _ = raw[picks]
        ch_names = [raw.ch_names[i] for i in picks]
        print(f"✅ Data shape: {data.shape}")

        print("⚡ Computing correlation matrix...")
        corr_matrix = np.corrcoef(data)

        mean_corrs = []
        for i in range(corr_matrix.shape[0]):
            mean_corr = (np.sum(corr_matrix[i]) - 1) / (corr_matrix.shape[0] - 1)
            mean_corrs.append(mean_corr)

        bad_channel_indices = [i for i, avg_corr in enumerate(mean_corrs) if avg_corr < correlation_threshold]
        bad_channels = [ch_names[i] for i in bad_channel_indices]

        print(f"\nSession: {session_file}")
        print(f"Total EEG channels: {len(ch_names)}")
        print(f"Detected bad channels (mean corr < {correlation_threshold}): {bad_channels}")

        plt.figure(figsize=(16, 14))
        plt.imshow(corr_matrix, cmap='viridis', interpolation='nearest')
        plt.colorbar(label='Correlation Coefficient')
        plt.title(f"Correlation Matrix - Subject {subj} {session_file}\n"
                  f"Potential bad channels: {', '.join(bad_channels) if bad_channels else 'None'}")
        plt.xticks(np.arange(len(ch_names)), ch_names, rotation=90, fontsize=6, ha='center')
        plt.yticks(np.arange(len(ch_names)), ch_names, fontsize=6)
        plt.xlabel("Channel")
        plt.ylabel("Channel")
        plt.tight_layout()
        plt.show(block=False)
        plt.pause(0.001)


# 4.Perform ICA - plot all components

In [ ]:
# Root EEG directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

def run_ica_excluding_breaks(participant):
    participant_path = os.path.join(root_dir, participant)
    fif_path = os.path.join(participant_path, f'{participant}_O1_filtered_raw.fif')
    ica_path = os.path.join(participant_path, f'{participant}_O1_ica.fif')

    if not os.path.exists(fif_path):
        print(f"[{participant}] ❌ No filtered .fif file found.")
        return
    if os.path.exists(ica_path):
        print(f"[{participant}] ✅ ICA already exists, skipping.")
        return

    print(f"\n>>> Running ICA for {participant} ")

    try:
        raw = mne.io.read_raw_fif(fif_path, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Crop to 10 minutes max to reduce memory load
        raw.crop(tmax=min(600, raw.times[-1]))

        # Mark 'break' annotations as bad segments
        if raw.annotations:
            break_indices = [i for i, desc in enumerate(raw.annotations.description) if desc.lower() == 'break']
            if break_indices:
                print(f"[{participant}] Found {len(break_indices)} 'break' annotations — excluding them from ICA.")
                raw.annotations.delete(break_indices)

        # Pick EEG channels only
        picks = mne.pick_types(raw.info, eeg=True, exclude='bads')

        # Run ICA
        ica = mne.preprocessing.ICA(n_components=None, method='fastica', random_state=97, max_iter=500)
        ica.fit(raw, picks=picks, reject_by_annotation=True)
        print(f"[{participant}] ✓ ICA fitted excluding breaks.")

        # Save ICA solution
        ica.save(ica_path, overwrite=True)
        print(f"[{participant}] ✓ ICA saved: {ica_path}")

        # Save ICA topomap in a single figure
        fig_list = ica.plot_components(inst=raw, show=False)
        fig = fig_list[0] if isinstance(fig_list, list) else fig_list
        fig_comp_path = os.path.join(participant_path, f"{participant}_O1_ica_components.png")
        fig.savefig(fig_comp_path, dpi=150)
        print(f"[{participant}] ✓ ICA component map saved.")

        # Save ICA time series
        fig_sources = ica.plot_sources(raw, show=False)
        fig_src_path = os.path.join(participant_path, f"{participant}_01_ica_sources.png")
        fig_sources.savefig(fig_src_path, dpi=150)
        print(f"[{participant}] ✓ ICA time series plot saved.")

        del raw, ica, fig, fig_sources
        gc.collect()

    except Exception as e:
        print(f"[{participant}] ⚠️ ERROR: {e}")
        traceback.print_exc()

# === Run for all participants ===
for pid in participants:
    run_ica_excluding_breaks(pid)


### EXCLUDE ICA COMPONENTS

In [ ]:
# Root directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Dict of components to exclude per participant
ica_removal_dict = {
    "3": [14, 15],
    "6": [1],
    "7": [],
    "8": [9, 13],
    "10": [15],
    "11": [],
    "12": [14],
    "13": [0, 5, 11, 12, 16],
    "14": [4],
    "15": [3, 6, 16],
    "17": [0, 5, 7, 10, 12, 16, 17],
    "18": [1, 2, 17],
    "19": [0, 1, 4, 7, 11, 12],
    "20": [],
    "21": [12, 17],
    "22": [1, 3, 14],
    "24": [1, 4, 8, 17],
    "25": [3, 15, 16],
    "26": [9],
    "27": [12]
}

# Loop through participants
for pid, comps in ica_removal_dict.items():
    print(f"\n>>> Processing participant {pid} | Exclude components: {comps}")
    participant_path = os.path.join(root_dir, pid)

    raw_path = os.path.join(participant_path, f"{pid}_O1_filtered_raw.fif")
    ica_path = os.path.join(participant_path, f"{pid}_O1_ica.fif")
    cleaned_path = os.path.join(participant_path, f"{pid}_O1_clean_after_ica.fif")

    # Skip if output already exists
    if os.path.exists(cleaned_path):
        print(f"[{pid}] ✅ Already cleaned — skipping.")
        continue

    if not os.path.exists(raw_path):
        print(f"[{pid}] ❌ Raw file not found: {raw_path}")
        continue
    if not os.path.exists(ica_path):
        print(f"[{pid}] ❌ ICA file not found: {ica_path}")
        continue

    try:
        # Load raw and ICA
        raw = mne.io.read_raw_fif(raw_path, preload=True)
        ica = mne.preprocessing.read_ica(ica_path)

        # Apply ICA exclusions
        ica.exclude = comps
        raw_clean = ica.apply(raw.copy())
        print(f"[{pid}] ✓ ICA applied")

        # Plot time series of sources (after ICA)
        fig_sources = ica.plot_sources(raw_clean, show=False)
        fig_path = os.path.join(participant_path, f"{pid}_O1_ica_sources_removed.png")
        fig_sources.savefig(fig_path, dpi=150)
        plt.close(fig_sources)
        print(f"[{pid}] ✓ Time series plot saved: {fig_path}")

        # Save cleaned raw
        raw_clean.save(cleaned_path, overwrite=True)
        print(f"[{pid}] ✓ Cleaned raw saved: {cleaned_path}")

        # Cleanup
        del raw, raw_clean, ica, fig_sources
        gc.collect()

    except Exception as e:
        print(f"[{pid}] ⚠️ Error: {e}")

# 5. Re-reference (LM/RM)


In [ ]:

# Root EEG directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Get numeric subject directories
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Processing Subject {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)

    # Find cleaned ICA files
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_O1_clean_after_ica.fif')])

    if not session_files:
        print(f"⚠️ No ICA-cleaned FIF files found for {subj}. Skipping.")
        continue

    print(f"🔍 Found ICA-cleaned files: {session_files}")

    for session_file in session_files:
        reref_file = session_file.replace('_O1_clean_after_ica.fif', '_O1_ica_reref_raw.fif')
        reref_path = os.path.join(subj_path, reref_file)

        # Check if already processed
        if os.path.exists(reref_path):
            print(f"✅ Already re-referenced: {reref_file} — skipping.")
            continue

        session_path = os.path.join(subj_path, session_file)
        print(f"📂 Loading: {session_path}")

        try:
            raw = mne.io.read_raw_fif(session_path, preload=True)

            # Check for mastoid channels
            if not all(ch in raw.ch_names for ch in ['E57', 'E100']):
                print(f"⚠️ Mastoid channels missing in {session_file}. Skipping.")
                continue

            # Apply linked mastoid reference
            lm = raw.copy().pick_channels(['E57']).get_data()
            rm = raw.copy().pick_channels(['E100']).get_data()
            mastoid_avg = (lm + rm) / 2
            raw._data = raw.get_data() - mastoid_avg

            print(f"✅ Re-referenced {session_file} using linked mastoids (E57/E100)")

            # Save re-referenced file
            raw.save(reref_path, overwrite=True)
            print(f"💾 Saved re-referenced file: {reref_path}")

        except Exception as e:
            print(f"❌ Error processing {session_file}: {e}")



# 6. Create Epochs


In [ ]:
# SET UP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne
import os
import mne
import traceback
import gc

In [ ]:
# Paths and constants
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

event_id = {'Wd2N': 1, 'Wd2E': 2}  # Normal and slowed
tmin, tmax = -0.1, 0.8  # Time window in seconds
baseline = (tmin, 0)  # Baseline correction period

def create_and_save_epochs(participant):
    participant_path = os.path.join(root_dir, participant)
    raw_file = os.path.join(participant_path, f"{participant}_O1_ica_reref_raw.fif")
    epo_file = os.path.join(participant_path, f"{participant}_O1_epo.fif")

    if not os.path.exists(raw_file):
        print(f"[{participant}] ❌ ICA reref file not found.")
        return

    if os.path.exists(epo_file):
        print(f"[{participant}] ✅ Epochs already exist, skipping.")
        return

    try:
        print(f"[{participant}] Creating epochs...")

        raw = mne.io.read_raw_fif(raw_file, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Convert annotations to events
        events, _ = mne.events_from_annotations(raw, event_id=event_id)

        # Epoching
        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_id,
            tmin=tmin,
            tmax=tmax,
            baseline=baseline,
            reject_by_annotation=True,
            preload=True
        )

        # Save epochs
        epochs.save(epo_file, overwrite=True)
        print(f"[{participant}] ✅ Epochs saved to {epo_file}")

        del raw, epochs
        gc.collect()

    except Exception as e:
        print(f"[{participant}] ⚠️ Error during epoching: {e}")
        traceback.print_exc()

# Run for all participants
for pid in participants:
    create_and_save_epochs(pid)

# 7. Artifacts Rejection - get the clean epochs


In [ ]:
#CHECK THE TRESHOLD
# === Configuration ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participant = "3"  # ← Change this or loop over more participants
epo_path = os.path.join(root_dir, participant, f"{participant}_O1_epo.fif")

# List of thresholds to test (in volts)
thresholds = [100e-6, 125e-6, 150e-6, 175e-6, 200e-6]
drop_rates = []

# === Load epochs ===
if not os.path.exists(epo_path):
    print(f"❌ Epoch file not found for participant {participant}")
else:
    print(f"📂 Loaded: {epo_path}")
    for th in thresholds:
        try:
            # Reload epochs fresh each time
            epochs = mne.read_epochs(epo_path, preload=True)
            n_total = len(epochs)

            # Apply threshold rejection
            epochs.drop_bad(reject=dict(eeg=th))
            n_clean = len(epochs)
            drop_rate = 1 - (n_clean / n_total)
            drop_rates.append(drop_rate)

            print(f"🔍 Threshold: {int(th*1e6)} µV — Dropped {drop_rate*100:.1f}% of epochs")

        except Exception as e:
            print(f"⚠️ Error at threshold {int(th*1e6)} µV: {e}")
            drop_rates.append(None)

    # === Plot results ===
    plt.figure(figsize=(8, 5))
    plt.plot([int(t*1e6) for t in thresholds], [r * 100 for r in drop_rates], marker='o')
    plt.xlabel('Threshold (µV)')
    plt.ylabel('Dropped Epochs (%)')
    plt.title(f'Epoch Rejection Rate — Participant {participant}')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

#ARIFACTS REJECTION
# Root directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Loop through participant folders
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print(f"\n🧠 Processing participant {subj}")
    subj_path = os.path.join(root_dir, subj)
    epo_path = os.path.join(subj_path, f"{subj}_O1_epo.fif")
    clean_epo_path = os.path.join(subj_path, f"{subj}_O1_epo_clean.fif")

    if not os.path.exists(epo_path):
        print(f"❌ No epoch file found for participant {subj}. Skipping.")
        continue

    if os.path.exists(clean_epo_path):
        print(f"✅ Cleaned epochs already exist for {subj}. Skipping.")
        continue

    try:
        epochs = mne.read_epochs(epo_path, preload=True)

        # Apply rejection: e.g., reject if any EEG channel exceeds ±150 µV
        epochs.drop_bad(reject=dict(eeg=150e-6))

        # Save cleaned epochs
        epochs.save(clean_epo_path, overwrite=True)
        print(f"💾 Cleaned epochs saved: {clean_epo_path}")

    except Exception as e:
        print(f"⚠️ Error processing participant {subj}: {e}")



# 8.Average across subjects → Grand average ERPs
Grand average ERP (all E trials)
Grand average ERP (all N trials)


In [ ]:
# SET UP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne
import os
import mne
import traceback
import gc
import matplotlib.pyplot as plt


In [ ]:
# Set root directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
output_dir = os.path.join(root_dir, 'Group_Level_Results')
os.makedirs(output_dir, exist_ok=True)

event_id = {'Wd2N': 1, 'Wd2E': 2}  # Normal context = Wd2N, Slowed context = Wd2E

# Containers for evoked responses per condition
evokeds_E = []
evokeds_N = []

# Loop through each participant (filtering for IDs >= 10)
participants = sorted([
    p for p in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, p)) and p.isdigit() and int(p) >= 10
])

for pid in participants:
    participant_path = os.path.join(root_dir, pid)
    epo_file = os.path.join(participant_path, f"{pid}_O1_epo_clean.fif")

    if not os.path.exists(epo_file):
        print(f"[{pid}] ❌ Clean epochs not found.")
        continue

    try:
        epochs = mne.read_epochs(epo_file, preload=True)
        print(f"[{pid}] Loaded cleaned epochs.")

        # Subject-level averaging
        if 'Wd2E' in epochs.event_id:
            evoked_E = epochs['Wd2E'].average()
            evokeds_E.append(evoked_E)
        if 'Wd2N' in epochs.event_id:
            evoked_N = epochs['Wd2N'].average()
            evokeds_N.append(evoked_N)

    except Exception as e:
        print(f"[{pid}] ⚠️ Error loading or processing: {e}")
        continue

# GRAND AVERAGES
if evokeds_E:
    grand_avg_E = mne.grand_average(evokeds_E)
    grand_avg_E.save(os.path.join(output_dir, 'new_O1_grand_average_E-ave.fif'))
    print("✅ Saved grand average ERP for Wd2E.")

if evokeds_N:
    grand_avg_N = mne.grand_average(evokeds_N)
    grand_avg_N.save(os.path.join(output_dir, 'new_O1_grand_average_N-ave.fif'))
    print("✅ Saved grand average ERP for Wd2N.")


# 9. Create 9 Region plots


In [ ]:
import os
import mne
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Load grand averages
grand_avg_E = mne.read_evokeds(os.path.join(output_dir, 'new_O1_grand_average_E-ave.fif'), condition=0)
grand_avg_N = mne.read_evokeds(os.path.join(output_dir, 'new_O1_grand_average_N-ave.fif'), condition=0)

# Define regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

region_order = [
    'Left Anterior', 'Medial Anterior', 'Right Anterior',
    'Left Central', 'Medial Central', 'Right Central',
    'Left Posterior', 'Medial Posterior', 'Right Posterior'
]

# Create figure
fig, axes = plt.subplots(3, 3, figsize=(14, 9), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.35, wspace=0.25, bottom=0.18)

styles = {
    'Normal Context': dict(color='black', linestyle='-'),
    'Slowed Context': dict(color='black', linestyle='--')
}

# Loop through regions
for i, region in enumerate(region_order):
    row, col = divmod(i, 3)
    ax = axes[row, col]

    picks = [ch for ch in regions[region] if ch in grand_avg_E.ch_names and ch in grand_avg_N.ch_names]
    if not picks:
        ax.set_visible(False)
        continue

    evoked_E_region = grand_avg_E.copy().pick_channels(picks)
    evoked_N_region = grand_avg_N.copy().pick_channels(picks)

    # Plot without individual legends
    mne.viz.plot_compare_evokeds(
        {'Slowed Context': evoked_E_region, 'Normal Context': evoked_N_region},
        picks=picks,
        axes=ax,
        styles=styles,
        show=False,
        split_legend=False,
        ci=0.68,
        combine='mean'
    )

    # Remove automatic subplot legend
    if ax.get_legend():
        ax.get_legend().remove()

    # Shade N100 and N280 windows
    ax.axvspan(0.100, 0.160, color='skyblue', alpha=0.3)
    ax.axvspan(0.240, 0.300, color='lightpink', alpha=0.3)

    ax.set_title(region.replace(" ", "\n"), fontsize=10)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Shared axis labels
fig.text(0.5, 0.06, 'Time (ms)', ha='center')
fig.text(0.06, 0.5, 'Amplitude (µV)', va='center', rotation='vertical')

# Global legend at bottom
line_norm = mlines.Line2D([], [], color='black', linestyle='-', label='Normal Context')
line_slow = mlines.Line2D([], [], color='black', linestyle='--', label='Slowed Context')
shade_n100 = mpatches.Patch(color='skyblue', alpha=0.3, label='N100 window')
shade_n280 = mpatches.Patch(color='lightpink', alpha=0.3, label='N280 window')

fig.legend(
    handles=[line_norm, line_slow, shade_n100, shade_n280],
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),
    ncol=4,
    frameon=False
)

plt.suptitle('Grand Average ERPs by Scalp Region', fontsize=14)
plt.savefig(os.path.join(output_dir, 'O1_new_group_ERP_topo_grid_cleaned_final.png'), dpi=300, bbox_inches='tight')
plt.show()


# 10. Statistical Analysis